In [1]:
import pandas as pd
from colbert.evaluation.evaluator import ColBERTEvaluator
from colbert.infra import ColBERTConfig


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/home/ec2-user/anaconda3/envs/JupyterSystemEnv/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/home/ec2-user/anaconda3/envs/JupyterSystemEnv/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/ec2-user/SageMaker/venvs/colbert/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/ec2-user/SageMaker/venvs/colbert/lib/python3.10/site-packages/traitlets

AttributeError: _ARRAY_API not found

In [2]:
config = ColBERTConfig(bsize=64, lr=1e-05, warmup=1000, doc_maxlen=512, dim=128, attend_to_mask_tokens=False, 
                       nway=2, accumsteps=2, similarity='cosine', use_ib_negatives=False)

In [3]:
evaluator=ColBERTEvaluator(config=config,checkpoint_path='./experiments/colbert_aspect_training/none/run_1741558706/checkpoints/colbert')

In [4]:
import os
from pathlib import Path
eval_out_path='ColBERT/experiments/colbert_aspect_training/run_1741558706/'
os.makedirs(eval_out_path,exist_ok=True)

In [5]:
from colbert.evaluation.triplet_loader import convert_triplets_to_qrels

In [6]:
queries_path='./experiments/colbert_aspect_training/run_1741558706/data/val/queries.train.colbert.tsv'
triplets_path='./experiments/colbert_aspect_training/run_1741558706/data/val/triples.train.colbert.jsonl'
qrels_path = convert_triplets_to_qrels(triplets_path,output_path=os.path.join(Path(queries_path).parent,'qrels.train.colbert.tsv')) if triplets_path else None

#> Loading triplets from ./experiments/colbert_aspect_training/run_1741558706/data/val/triples.train.colbert.jsonl
#> Loaded 1001 queries with positives
#> Average positives per query: 8.20
#> Average negatives per query: 30.19
#> Converted triplets to qrels format: experiments/colbert_aspect_training/run_1741558706/data/val/qrels.train.colbert.tsv


In [7]:
collection_path='./experiments/colbert_aspect_training/run_1741558706/data/val/corpus.train.colbert.tsv'

In [3]:
eval_out_path

NameError: name 'eval_out_path' is not defined

In [ ]:
eval_res=evaluator.evaluate(
        queries_path=queries_path,
        collection_path=collection_path,
        triplets_path=triplets_path,
        output_path=os.path.join(eval_out_path,'output.train.colbert.tsv'),
        batch_size=2048,
        depth=100,
        step=None
    )

[Mar 10, 06:57:29] #> Loading the queries from ./experiments/colbert_aspect_training/run_1741558706/data/val/queries.train.colbert.tsv ...
[Mar 10, 06:57:29] #> Got 1057 queries. All QIDs are unique.

[Mar 10, 06:57:29] #> Loading collection...
0M 
#> Loading triplets from ./experiments/colbert_aspect_training/run_1741558706/data/val/triples.train.colbert.jsonl
#> Loaded 1001 queries with positives
#> Average positives per query: 8.20
#> Average negatives per query: 30.19
#> Converted triplets to qrels format: experiments/colbert_aspect_training/run_1741558706/data/val/qrels.train.colbert.tsv
[Mar 10, 06:57:30] #> Loading qrels from experiments/colbert_aspect_training/run_1741558706/data/val/qrels.train.colbert.tsv ...
[Mar 10, 06:57:30] #> Loaded qrels for 1001 unique queries with 8.2 positives per query on average.

[Mar 10, 06:57:30] #> Processing query 1 / 1057

#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: Customer Industry - Nutraceuticals in

  4%|▍         | 1/26 [00:22<09:23, 22.55s/it]

In [ ]:
eval_res

In [3]:
def adaptive_binary_search(scores, target_precision_range, default_cutoff_index=None, k=5, precision_checker=None, max_iterations=100):
    """
    Performs an adaptive binary search to find a cutoff index where precision falls within a target range.
    
    Args:
        scores: Sorted array of cosine similarity scores (assumed to be in descending order)
        target_precision_range: Tuple of (min_precision, max_precision) defining the acceptable range
        default_cutoff_index: Starting index for the search (defaults to middle of array)
        k: Number of elements to include on each side for precision evaluation
        precision_checker: Function that takes a subset of scores around cutoff and returns precision
        max_iterations: Maximum number of iterations to prevent infinite loops
    
    Returns:
        The index representing the optimal cutoff position
    """
    if not scores:
        return None
    
    # Set default cutoff to middle of array if not provided
    if default_cutoff_index is None:
        default_cutoff_index = len(scores) // 2
    
    # Initialize search bounds
    left = 0
    right = len(scores) - 1
    current_index = default_cutoff_index
    
    # Define min and max precision targets
    min_precision, max_precision = target_precision_range
    
    # Define minimum jump size
    min_jump_size = 1
    
    iterations = 0
    
    while iterations < max_iterations:
        iterations += 1
        
        # Get current precision
        window_start = max(0, current_index - k)
        window_end = min(len(scores) - 1, current_index + k)
        current_window = scores[window_start:window_end + 1]
        current_precision = precision_checker(current_window, current_index - window_start)
        
        # Check if we've reached target precision
        if min_precision <= current_precision <= max_precision:
            return current_index
        
        # Calculate how far we are from the target range
        if current_precision < min_precision:
            # We need to increase precision (likely move toward higher scores)
            distance_from_target = min_precision - current_precision
            direction = -1  # Move left (assuming scores are in descending order)
            remaining_distance = current_index - left
        else:  # current_precision > max_precision
            # We need to decrease precision (likely move toward lower scores)
            distance_from_target = current_precision - max_precision
            direction = 1  # Move right (assuming scores are in descending order)
            remaining_distance = right - current_index
        
        # Calculate adaptive jump size based on distance from target
        # The further from target precision range, the bigger the jump
        # Use a normalized factor based on distance from target
        normalized_factor = min(1.0, distance_from_target)  # Cap at 1.0
        jump_size = max(min_jump_size, int(remaining_distance * normalized_factor))
        
        # Ensure we're making progress
        if jump_size < min_jump_size:
            jump_size = min_jump_size
        
        # Calculate new index
        new_index = current_index + (direction * jump_size)
        
        # Ensure new index is within bounds
        new_index = max(left, min(right, new_index))
        
        # Break if we're not making progress
        if new_index == current_index:
            break
            
        # Update current index and search bounds
        current_index = new_index
        if direction < 0:  # Moving left
            right = min(right, current_index + jump_size)
        else:  # Moving right
            left = max(left, current_index - jump_size)
    
    # If we've exhausted iterations, return the best index we found
    return current_index

In [4]:
def sample_precision_checker(window_scores, center_position):
    """
    Example precision checker that calculates a simulated precision value
    based on the scores around the cutoff.
    
    In a real scenario, this would compute actual precision based on
    the model's outputs for the subset of data.
    """
    # Simple example: higher scores above center give better precision
    above_cutoff = window_scores[:center_position]
    total_score = sum(above_cutoff) if above_cutoff else 0
    
    # Normalize to a 0-1 range
    return min(1.0, total_score / len(window_scores) if window_scores else 0)

# Example usage
scores = [0.95, 0.92, 0.87, 0.82, 0.78, 0.75, 0.70, 0.65, 0.60, 0.55, 0.50, 0.45, 0.40]
target_range = (0.6, 0.7)  # We want precision between 60-70%

cutoff_index = adaptive_binary_search(
    scores=scores,
    target_precision_range=target_range,
    precision_checker=sample_precision_checker,
    k=3
)

print(f"Optimal cutoff index: {cutoff_index}, Score: {scores[cutoff_index]}")

Optimal cutoff index: 0, Score: 0.95
